In [2]:
!pip install -q groq requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 5.4 MB/s eta 0:00:00


In [ ]:
import json
import unicodedata
import requests
from groq import Groq
from google.colab import userdata

# ── Configuração ──────────────────────────────────────────────────────────────

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
TMDB_API_KEY = userdata.get('TMDB_API_KEY')

client = Groq(api_key=GROQ_API_KEY)

GENRES = {
    16: "Animação",   35: "Comédia",        10751: "Família",
    14: "Fantasia",   18: "Drama",           10749: "Romance",
    99: "Documentário", 12: "Aventura",      28: "Ação",
    27: "Terror",    878: "Ficção Científica", 80: "Crime",
  9648: "Mistério",   53: "Suspense",       10752: "Guerra",
    37: "Faroeste",   36: "História",        10402: "Música",
 10770: "Telefilme",
}

STOPWORDS = {
    "o", "a", "os", "as", "um", "uma", "de", "do", "da", "dos", "das",
    "e", "em", "no", "na", "nos", "nas", "por", "para", "com", "que",
    "the", "an", "of", "in", "on", "at", "to", "and", "is", "it",
}

# ── Utilitários ───────────────────────────────────────────────────────────────

def normalizar(texto: str) -> str:
    """Minúsculas, sem acentos e sem espaços extras — para comparação segura."""
    sem_acento = unicodedata.normalize("NFD", texto)
    sem_acento = "".join(c for c in sem_acento if unicodedata.category(c) != "Mn")
    return " ".join(sem_acento.lower().strip().split())


def palavras_chave(texto: str) -> list[str]:
    """Palavras significativas: mais de 3 caracteres e fora das stopwords."""
    return [p for p in normalizar(texto).split() if len(p) > 3 and p not in STOPWORDS]


# ── IA ────────────────────────────────────────────────────────────────────────

def obter_filmes_da_ia(prompt_usuario: str, historico: list[str]) -> dict:
    """Solicita 3 filmes reais à IA. Retorna {'mensagem': str, 'lista_filmes': list}."""

    proibidos = ""
    if historico:
        proibidos = (
            "\n\n### FILMES PROIBIDOS — VOCE JA SUGERIU ESSES, NAO REPITA NENHUM:\n"
            + "\n".join(f"- {f}" for f in historico)
        )

    sistema = (
        "Você é um especialista em cinema. Sua UNICA função é indicar filmes REAIS "
        "que existam no IMDb e no TMDB com mais de 50.000 avaliações de usuários.\n\n"

        "### REGRAS ABSOLUTAS — NUNCA VIOLE:\n"
        "1. NUNCA invente títulos. Se não tiver 100% de certeza que o filme existe, NAO sugira.\n"
        "2. NUNCA altere o ano de lançamento de um filme.\n"
        "3. NUNCA combine partes de títulos diferentes para criar um nome novo.\n"
        "4. NUNCA adicione subtítulos que não existem no título oficial.\n"
        "5. SEMPRE use a grafia correta e completa do título — ex: 'Interstellar', não 'Interstelar'.\n"
        "6. Use apenas filmes amplamente conhecidos, com nota IMDb >= 7.0 e mais de 50.000 votos.\n"
        "7. NUNCA repita filmes da lista de proibidos, nem com grafia ou subtítulo diferente.\n"
        "8. Sugira EXATAMENTE 3 filmes — nem mais, nem menos.\n"
        "9. Os filmes DEVEM combinar com o humor/sentimento descrito pelo usuário. "
        "Se pede 'algo alegre', sugira comédias ou filmes leves. "
        "Se pede 'algo triste', sugira dramas emocionantes. Respeite o pedido.\n\n"

        "### FORMATO DE RESPOSTA OBRIGATORIO:\n"
        "Responda SOMENTE com este JSON, sem nada antes ou depois:\n"
        '{"mensagem": "frase curta e animada (max 10 palavras)", '
        '"lista_filmes": ["Título Exato 1", "Título Exato 2", "Título Exato 3"]}\n\n'

        "### REGRAS CRITICAS DE FORMATO:\n"
        "- SEM texto antes ou depois do JSON\n"
        "- SEM explicações ou comentários\n"
        "- SEM backticks ou markdown\n"
        "- SEM a palavra output\n"
        "- APENAS o JSON valido"
        + proibidos
    )

    resposta = client.chat.completions.create(
        messages=[
            {"role": "system",  "content": sistema},
            {"role": "user",    "content": prompt_usuario},
        ],
        model="llama-3.3-70b-versatile",
        temperature=0.6,
        response_format={"type": "json_object"},
    )

    dados = json.loads(resposta.choices[0].message.content)

    if "lista_filmes" not in dados or not isinstance(dados["lista_filmes"], list):
        raise ValueError(f"Resposta fora do formato esperado: {dados}")

    return dados


# ── TMDB ──────────────────────────────────────────────────────────────────────

def _buscar_validado(nome: str) -> dict | None:
    """Busca e valida um filme no TMDB: >= 500 votos e título com correspondência."""
    url = (
        "https://api.themoviedb.org/3/search/movie"
        f"?api_key={TMDB_API_KEY}&query={requests.utils.quote(nome)}&language=pt-BR"
    )
    try:
        resultados = requests.get(url, timeout=8).json().get("results", [])[:5]
        chaves = palavras_chave(nome)
        candidatos = [
            f for f in resultados
            if f.get("vote_count", 0) >= 500
            and (not chaves or any(
                p in normalizar(f.get("title", "")) or
                p in normalizar(f.get("original_title", ""))
                for p in chaves
            ))
        ]
        return max(candidatos, key=lambda f: f.get("vote_count", 0)) if candidatos else None
    except Exception:
        return None


def _corrigir_grafia(nome: str) -> str:
    """
    Tenta corrigir erros de grafia via TMDB (ex: 'Interstelar' → 'Interstellar').
    Retorna o título original oficial se encontrar correspondência com > 10.000 votos.
    """
    url = (
        "https://api.themoviedb.org/3/search/movie"
        f"?api_key={TMDB_API_KEY}&query={requests.utils.quote(nome)}&language=en-US"
    )
    try:
        chaves = palavras_chave(nome)
        for filme in requests.get(url, timeout=8).json().get("results", [])[:3]:
            if filme.get("vote_count", 0) < 10_000:
                continue
            titulo_orig = normalizar(filme.get("original_title", ""))
            if chaves and any(p in titulo_orig for p in chaves):
                return filme.get("original_title", nome)
    except Exception:
        pass
    return nome


def buscar_detalhes_tmdb(nome: str) -> dict | None:
    """Retorna detalhes do filme validado; tenta corrigir grafia se a busca falhar."""
    resultado = _buscar_validado(nome)
    if resultado:
        return resultado

    corrigido = _corrigir_grafia(nome)
    if normalizar(corrigido) != normalizar(nome):
        return _buscar_validado(corrigido)

    return None


def buscar_streaming(movie_id: int) -> list[str]:
    """Retorna plataformas de streaming disponíveis no Brasil."""
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/watch/providers?api_key={TMDB_API_KEY}"
    try:
        br = requests.get(url, timeout=8).json().get("results", {}).get("BR", {})
        if br.get("flatrate"):
            return [p["provider_name"] for p in br["flatrate"]]
        if br.get("rent"):
            return ["Aluguel: " + ", ".join(p["provider_name"] for p in br["rent"])]
        return ["Não disponível no streaming BR"]
    except Exception:
        return ["Erro ao buscar streaming"]


# ── Chat interativo ───────────────────────────────────────────────────────────

def chatbot_interativo():
    print("🎬 Bem-vindo ao seu cinema particular! (Digite 'sair' para encerrar)\n")

    historico:      list[str] = []
    historico_norm: set[str]  = set()

    while True:
        user_input = input("Você: ").strip()

        if not user_input:
            continue

        if user_input.lower() in {"sair", "exit", "quit"}:
            print("\n🎬 Espero que você aproveite o filme! Até a próxima. ✨")
            break

        try:
            print("\n⏳ Consultando o curador...")

            sugestao = obter_filmes_da_ia(user_input, historico)
            print(f"\n✨ {sugestao['mensagem']}\n")

            exibidos = 0

            for nome in sugestao["lista_filmes"]:
                if normalizar(nome) in historico_norm:
                    continue

                historico.append(nome)
                historico_norm.add(normalizar(nome))

                filme = buscar_detalhes_tmdb(nome)
                if not filme:
                    print(f"   ⚠️  '{nome}' não confirmado no TMDB — descartado.\n")
                    continue

                generos  = [GENRES.get(g, "Geral") for g in filme.get("genre_ids", [])[:2]]
                ano      = filme.get("release_date", "????")[:4]
                nota     = filme.get("vote_average", 0)
                votos    = filme.get("vote_count", 0)
                overview = filme.get("overview", "Sem sinopse disponível.")
                streams  = buscar_streaming(filme["id"])

                print(f"🎥 {filme['title']} ({ano})")
                print(f"   {' · '.join(generos) or 'Geral'} | ⭐ {nota:.1f}  ({votos:,} votos)")
                print(f"   Onde assistir: {', '.join(streams)}")
                print(f"   {overview[:160]}{'...' if len(overview) > 160 else ''}\n")
                exibidos += 1

            if exibidos == 0:
                print("⚠️  Nenhum filme passou na validação. Tente um pedido diferente.\n")
            else:
                print("-" * 40)
                print(f"📋 Histórico: {len(historico)} filme(s) — não serão repetidos.")
                print("👉 Refine se quiser: 'quero animações', 'algo mais antigo', 'terror leve'\n")

        except json.JSONDecodeError:
            print("[Erro] A IA retornou formato inválido. Tente novamente.\n")
        except Exception as e:
            print(f"[Erro] {e}\n")


chatbot_interativo()